# GMM Clustering — Probabilistic Customer Segmentation (HIGHLIGHT)

**Project:** E-commerce Sales Data Analysis & Machine Learning  
**Author:** Muhammad Iqbal Fadel  
**Source:** `scripts/07_gmm_clustering.py`

## Overview

Gaussian Mixture Model (GMM) Clustering Pipeline
Highlight utama project Data Mining - E-commerce Customer Segmentation

Pipeline:
Step 1: Data Preparation & Scaling
Step 2: Optimal Cluster Selection (BIC/AIC)
Step 3: GMM Training & Assignment
Step 4: Cluster Profiling & Labeling
Step 5: Visualisasi (7 plot)
Step 6: GMM vs RFM Comparison
Step 7: Business Recommendations

Author: Muhammad Iqbal Fadel
Date: May 2026

In [ ]:
%matplotlib inline

from __future__ import annotations

import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ── Paths ──────────────────────────────────────────────────────────────────

In [ ]:
ROOT = os.path.abspath('..')
CUSTOMER_DATA = os.path.join(ROOT, 'data', 'processed', 'model_data_customers.csv')
CLEANED_DATA = os.path.join(ROOT, 'data', 'processed', 'Sales_Transaction_v4a_cleaned.csv')

FIG_DIR = os.path.join(ROOT, 'outputs', 'figures', 'gmm')
DATA_DIR = os.path.join(ROOT, 'outputs', 'data', 'gmm')
RFM_CSV = os.path.join(ROOT, 'outputs', 'data', 'analysis', 'rfm_summary.csv')

for d in (FIG_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)


def savefig(name: str) -> None:
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'  [OK] Saved: {path}')

## STEP 1: DATA PREPARATION

In [ ]:
def step1_prepare_data() -> tuple[pd.DataFrame, np.ndarray, StandardScaler, list[str]]:
    """Load customer data, select features, handle outliers, scale."""
    print('\n' + '='*60)
    print('STEP 1: Data Preparation')
    print('='*60)

    df = pd.read_csv(CUSTOMER_DATA)
    print(f'  Loaded {len(df)} customers from model_data_customers.csv')

    feature_cols = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'DistinctProducts']
    X_raw = df[feature_cols].copy()

    # Handle inf/nan
    X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Clip outliers using IQR method (cap at 1.5*IQR)
    for col in feature_cols:
        Q1 = X_raw[col].quantile(0.25)
        Q3 = X_raw[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        X_raw[col] = X_raw[col].clip(lower=lower, upper=upper)

    # StandardScaler normalization
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    print(f'  Features: {feature_cols}')
    print(f'  Shape after scaling: {X_scaled.shape}')
    print(f'  Outliers clipped using IQR method')

    return df, X_scaled, scaler, feature_cols

## STEP 2: OPTIMAL CLUSTER SELECTION (BIC / AIC)

In [ ]:
def step2_find_optimal_k(X: np.ndarray, k_range: range = range(2, 11)) -> int:
    """Find optimal number of clusters using BIC and AIC."""
    print('\n' + '='*60)
    print('STEP 2: Optimal Cluster Selection (BIC/AIC)')
    print('='*60)

    bic_scores = []
    aic_scores = []

    for k in k_range:
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            random_state=42,
            n_init=5,
            max_iter=300
        )
        gmm.fit(X)
        bic_scores.append(gmm.bic(X))
        aic_scores.append(gmm.aic(X))
        print(f'  k={k}: BIC={gmm.bic(X):.1f}, AIC={gmm.aic(X):.1f}')

    # Optimal k = minimum BIC
    optimal_k = list(k_range)[np.argmin(bic_scores)]
    print(f'\n  >> Optimal k (min BIC): {optimal_k}')

## ── Plot BIC/AIC ──

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(list(k_range), bic_scores, 'o-', color='#2196F3', linewidth=2.5,
            markersize=8, label='BIC', zorder=5)
    ax.plot(list(k_range), aic_scores, 's--', color='#FF5722', linewidth=2.5,
            markersize=8, label='AIC', zorder=5)
    ax.axvline(x=optimal_k, color='#4CAF50', linestyle=':', linewidth=2,
               label=f'Optimal k = {optimal_k}')
    ax.fill_betweenx(ax.get_ylim(), optimal_k - 0.3, optimal_k + 0.3,
                     alpha=0.15, color='#4CAF50')
    ax.set_xlabel('Number of Clusters (k)')
    ax.set_ylabel('Information Criterion Score')
    ax.set_title('GMM Cluster Selection: BIC & AIC Scores')
    ax.legend(fontsize=12, loc='upper right')
    ax.set_xticks(list(k_range))
    ax.grid(True, alpha=0.3)
    savefig('gmm_bic_aic_selection.png')

    return optimal_k

## STEP 3: GMM TRAINING & ASSIGNMENT

In [ ]:
def step3_train_gmm(X: np.ndarray, k: int) -> tuple[GaussianMixture, np.ndarray, np.ndarray]:
    """Train final GMM and assign clusters."""
    print('\n' + '='*60)
    print(f'STEP 3: GMM Training (k={k})')
    print('='*60)

    gmm = GaussianMixture(
        n_components=k,
        covariance_type='full',
        random_state=42,
        n_init=10,
        max_iter=500,
        tol=1e-4
    )
    gmm.fit(X)

    labels = gmm.predict(X)
    probabilities = gmm.predict_proba(X)

    print(f'  Converged: {gmm.converged_}')
    print(f'  Iterations: {gmm.n_iter_}')
    print(f'  Log-likelihood: {gmm.score(X):.4f}')
    print(f'  Cluster distribution:')
    for i in range(k):
        count = (labels == i).sum()
        pct = count / len(labels) * 100
        print(f'    Cluster {i}: {count} customers ({pct:.1f}%)')

    # Save model
    model_path = os.path.join(DATA_DIR, 'gmm_model.pkl')
    joblib.dump(gmm, model_path)
    print(f'  [OK] Model saved: {model_path}')

    return gmm, labels, probabilities

## STEP 4: CLUSTER PROFILING & LABELING

In [ ]:
def step4_profile_clusters(
    df: pd.DataFrame, labels: np.ndarray, feature_cols: list[str]
) -> pd.DataFrame:
    """Profile each cluster and assign descriptive labels."""
    print('\n' + '='*60)
    print('STEP 4: Cluster Profiling & Labeling')
    print('='*60)

    df = df.copy()
    df['GMM_Cluster'] = labels

    # Compute cluster means for key features
    profile = df.groupby('GMM_Cluster').agg(
        Count=('GMM_Cluster', 'size'),
        Avg_Recency=('Recency', 'mean'),
        Avg_Frequency=('Frequency', 'mean'),
        Avg_Monetary=('Monetary', 'mean'),
        Avg_OrderValue=('AvgOrderValue', 'mean'),
        Avg_DistinctProducts=('DistinctProducts', 'mean'),
    ).reset_index()

    # Auto-label clusters based on characteristics
    def assign_label(row, all_profiles):
        """Assign descriptive label based on relative position in all clusters."""
        monetary_rank = (all_profiles['Avg_Monetary'] <= row['Avg_Monetary']).sum()
        frequency_rank = (all_profiles['Avg_Frequency'] <= row['Avg_Frequency']).sum()
        recency_rank = (all_profiles['Avg_Recency'] <= row['Avg_Recency']).sum()
        n = len(all_profiles)

        # High monetary + high frequency + low recency = best
        if monetary_rank >= n * 0.75 and frequency_rank >= n * 0.75:
            return 'High-Value Loyal'
        elif monetary_rank >= n * 0.5 and frequency_rank >= n * 0.5:
            return 'Mid-Value Regular'
        elif recency_rank >= n * 0.75:
            return 'At-Risk / Dormant'
        elif monetary_rank <= n * 0.25:
            return 'Low-Value Occasional'
        else:
            return 'Emerging / Potential'

    profile['Cluster_Label'] = profile.apply(
        lambda row: assign_label(row, profile), axis=1
    )

    # If duplicate labels exist, append cluster number
    label_counts = profile['Cluster_Label'].value_counts()
    duplicates = label_counts[label_counts > 1].index
    if len(duplicates) > 0:
        for dup_label in duplicates:
            mask = profile['Cluster_Label'] == dup_label
            indices = profile[mask].index
            for i, idx in enumerate(indices):
                profile.loc[idx, 'Cluster_Label'] = f'{dup_label} {i+1}'

    profile['Pct'] = (profile['Count'] / profile['Count'].sum() * 100).round(1)

    print(profile[['GMM_Cluster', 'Cluster_Label', 'Count', 'Pct',
                    'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']].to_string(index=False))

    profile_path = os.path.join(DATA_DIR, 'gmm_cluster_profiles.csv')
    profile.to_csv(profile_path, index=False)
    print(f'\n  [OK] Saved: {profile_path}')

    # Save assignments
    assignments = df[['CustomerNo', 'GMM_Cluster']].copy()
    assignments = assignments.merge(
        profile[['GMM_Cluster', 'Cluster_Label']], on='GMM_Cluster', how='left'
    )
    assign_path = os.path.join(DATA_DIR, 'gmm_cluster_assignments.csv')
    assignments.to_csv(assign_path, index=False)
    print(f'  [OK] Saved: {assign_path}')

    return profile

## STEP 5: VISUALIZATIONS

In [ ]:
def step5_visualize(
    df: pd.DataFrame, X_scaled: np.ndarray, labels: np.ndarray,
    probabilities: np.ndarray, profile: pd.DataFrame, feature_cols: list[str]
) -> None:
    """Generate all GMM visualizations."""
    print('\n' + '='*60)
    print('STEP 5: Generating Visualizations')
    print('='*60)

    k = len(profile)
    label_map = dict(zip(profile['GMM_Cluster'], profile['Cluster_Label']))
    colors = sns.color_palette('husl', k)

## ── 5a. PCA 2D Scatter Plot ──────────────────────────────────────────

In [ ]:
pca2 = PCA(n_components=2)
    X_2d = pca2.fit_transform(X_scaled)
    var_explained = pca2.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(11, 8))
    for i in range(k):
        mask = labels == i
        ax.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            c=[colors[i]], label=f'{label_map[i]} (n={mask.sum()})',
            alpha=0.6, s=30, edgecolors='white', linewidth=0.3
        )
    ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}% variance)')
    ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}% variance)')
    ax.set_title('GMM Customer Clusters - PCA 2D Projection')
    ax.legend(fontsize=10, loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.2)
    savefig('gmm_cluster_scatter_2d.png')

## ── 5b. PCA 3D Scatter Plot ──────────────────────────────────────────

In [ ]:
pca3 = PCA(n_components=3)
    X_3d = pca3.fit_transform(X_scaled)
    var3 = pca3.explained_variance_ratio_

    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')
    for i in range(k):
        mask = labels == i
        ax.scatter(
            X_3d[mask, 0], X_3d[mask, 1], X_3d[mask, 2],
            c=[colors[i]], label=label_map[i],
            alpha=0.5, s=20
        )
    ax.set_xlabel(f'PC1 ({var3[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({var3[1]*100:.1f}%)')
    ax.set_zlabel(f'PC3 ({var3[2]*100:.1f}%)')
    ax.set_title('GMM Customer Clusters - PCA 3D Projection')
    ax.legend(fontsize=9, loc='upper left')
    savefig('gmm_cluster_scatter_3d.png')

## ── 5c. Cluster Distribution Bar Chart ───────────────────────────────

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(
        profile['Cluster_Label'], profile['Count'],
        color=colors[:k], edgecolor='white', linewidth=1.5
    )
    for bar, pct in zip(bars, profile['Pct']):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{int(bar.get_height())}\n({pct}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Number of Customers')
    ax.set_title('GMM Cluster Distribution')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    savefig('gmm_cluster_distribution.png')

## ── 5d. Radar / Spider Chart ─────────────────────────────────────────

In [ ]:
radar_cols = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary',
                  'Avg_OrderValue', 'Avg_DistinctProducts']
    radar_labels = ['Recency\n(lower=better)', 'Frequency', 'Monetary',
                    'Avg Order\nValue', 'Distinct\nProducts']

    # Normalize to 0-1 for radar
    radar_data = profile[radar_cols].copy()
    for col in radar_cols:
        min_val = radar_data[col].min()
        max_val = radar_data[col].max()
        if max_val > min_val:
            radar_data[col] = (radar_data[col] - min_val) / (max_val - min_val)
        else:
            radar_data[col] = 0.5

    angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    for i in range(k):
        values = radar_data.iloc[i].tolist()
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=label_map[i], color=colors[i])
        ax.fill(angles, values, alpha=0.1, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=10)
    ax.set_title('GMM Cluster Profiles - Radar Chart', size=14, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    savefig('gmm_cluster_profiles_radar.png')

## ── 5e. Probability Heatmap ──────────────────────────────────────────

In [ ]:
# Show average membership probability per assigned cluster
    prob_df = pd.DataFrame(
        probabilities,
        columns=[label_map[i] for i in range(k)]
    )
    prob_df['Assigned'] = [label_map[l] for l in labels]

    prob_matrix = prob_df.groupby('Assigned').mean()
    # Reorder to match profile order
    ordered_labels = [label_map[i] for i in range(k)]
    prob_matrix = prob_matrix.reindex(ordered_labels)
    prob_matrix = prob_matrix[ordered_labels]

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(
        prob_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
        linewidths=1, linecolor='white', ax=ax,
        vmin=0, vmax=1,
        cbar_kws={'label': 'Avg. Membership Probability'}
    )
    ax.set_title('GMM Membership Probability Heatmap')
    ax.set_xlabel('Cluster (Probability of Belonging)')
    ax.set_ylabel('Assigned Cluster')
    savefig('gmm_probability_heatmap.png')

## STEP 6: GMM vs RFM COMPARISON

In [ ]:
def step6_compare_rfm(df: pd.DataFrame, labels: np.ndarray, profile: pd.DataFrame) -> None:
    """Compare GMM clusters with RFM segments."""
    print('\n' + '='*60)
    print('STEP 6: GMM vs RFM Comparison')
    print('='*60)

    label_map = dict(zip(profile['GMM_Cluster'], profile['Cluster_Label']))

    # Rebuild RFM segments from customer data
    df = df.copy()
    df['GMM_Cluster_Label'] = [label_map[l] for l in labels]

    # Need original transaction data for RFM
    if not os.path.exists(CLEANED_DATA):
        print('  [WARN] Cleaned data not found, skipping RFM comparison')
        return

    txn = pd.read_csv(CLEANED_DATA, low_memory=False)
    txn['Date'] = pd.to_datetime(txn['Date'], errors='coerce')
    if 'TotalAmount' not in txn.columns:
        txn['TotalAmount'] = txn['Price'] * txn['Quantity']

    rfm_base = txn.dropna(subset=['CustomerNo', 'Date', 'TotalAmount']).copy()
    ref_date = rfm_base['Date'].max() + pd.Timedelta(days=1)
    freq_col = 'TransactionNo' if 'TransactionNo' in rfm_base.columns else 'Date'
    rfm = rfm_base.groupby('CustomerNo').agg(
        Recency=('Date', lambda x: (ref_date - x.max()).days),
        Frequency=(freq_col, 'nunique'),
        Monetary=('TotalAmount', 'sum')
    ).reset_index()

    rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 4, labels=[4, 3, 2, 1]).astype(int)
    rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
    rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)

    def segment_customer(row):
        if row['R_Score'] >= 3 and row['F_Score'] >= 3 and row['M_Score'] >= 3:
            return 'Champions'
        if row['R_Score'] >= 3 and row['F_Score'] >= 2:
            return 'Loyal'
        if row['R_Score'] <= 2 and row['F_Score'] >= 3:
            return 'At Risk'
        if row['R_Score'] <= 2 and row['F_Score'] <= 2:
            return 'Hibernating'
        return 'Potential'

    rfm['RFM_Segment'] = rfm.apply(segment_customer, axis=1)

    # Merge GMM labels with RFM segments
    merged = rfm[['CustomerNo', 'RFM_Segment']].merge(
        df[['CustomerNo', 'GMM_Cluster_Label']], on='CustomerNo', how='inner'
    )

    if merged.empty:
        print('  [WARN] No matching customers found')
        return

    # Cross-tabulation
    cross = pd.crosstab(
        merged['GMM_Cluster_Label'], merged['RFM_Segment'],
        margins=True, margins_name='Total'
    )
    print('\n  Cross-tabulation (GMM vs RFM):')
    print(cross.to_string())

    # Heatmap (without margins)
    cross_no_margin = pd.crosstab(merged['GMM_Cluster_Label'], merged['RFM_Segment'])

    fig, ax = plt.subplots(figsize=(11, 7))
    sns.heatmap(
        cross_no_margin, annot=True, fmt='d', cmap='Blues',
        linewidths=1, linecolor='white', ax=ax,
        cbar_kws={'label': 'Number of Customers'}
    )
    ax.set_title('GMM Clusters vs RFM Segments - Cross-tabulation')
    ax.set_xlabel('RFM Segment')
    ax.set_ylabel('GMM Cluster')
    savefig('gmm_vs_rfm_comparison.png')

    print(f'\n  Total matched customers: {len(merged)}')

## STEP 7: BUSINESS RECOMMENDATIONS

In [ ]:
def step7_recommendations(profile: pd.DataFrame) -> None:
    """Generate business recommendations per cluster."""
    print('\n' + '='*60)
    print('STEP 7: Business Recommendations')
    print('='*60)

    recommendations = []
    for _, row in profile.iterrows():
        label = row['Cluster_Label']

        if 'High-Value' in label or 'Loyal' in label:
            action = (
                'Pertahankan dengan loyalty program eksklusif, early access produk baru, '
                'dan personalized offers. Prioritaskan customer service premium.'
            )
            priority = 'Tinggi'
        elif 'Mid-Value' in label or 'Regular' in label:
            action = (
                'Tingkatkan engagement dengan cross-selling dan bundling. '
                'Berikan insentif untuk meningkatkan frekuensi pembelian.'
            )
            priority = 'Tinggi'
        elif 'At-Risk' in label or 'Dormant' in label:
            action = (
                'Jalankan win-back campaign dengan diskon terbatas dan reminder produk favorit. '
                'Kirim survey untuk memahami alasan penurunan aktivitas.'
            )
            priority = 'Sedang'
        elif 'Low-Value' in label or 'Occasional' in label:
            action = (
                'Dorong pembelian kedua dengan voucher selamat datang dan rekomendasi produk populer. '
                'Gunakan email marketing untuk membangun kebiasaan belanja.'
            )
            priority = 'Sedang'
        else:  # Emerging / Potential
            action = (
                'Berikan pengalaman belanja terbaik untuk konversi. Tawarkan bundling starter, '
                'free shipping, dan highlight produk best-seller.'
            )
            priority = 'Sedang'

        recommendations.append({
            'Cluster': row['GMM_Cluster'],
            'Label': label,
            'Customer_Count': int(row['Count']),
            'Pct': row['Pct'],
            'Avg_Monetary': round(row['Avg_Monetary'], 2),
            'Recommended_Action': action,
            'Priority': priority
        })

    rec_df = pd.DataFrame(recommendations)
    rec_path = os.path.join(DATA_DIR, 'gmm_strategy_recommendations.csv')
    rec_df.to_csv(rec_path, index=False)
    print(rec_df[['Label', 'Customer_Count', 'Priority', 'Recommended_Action']].to_string(index=False))
    print(f'\n  [OK] Saved: {rec_path}')

In [ ]:
# MAIN PIPELINE

In [ ]:
def main():
    print('=' * 60)
    print('   GMM CLUSTERING PIPELINE - E-commerce Customer Seg.')
    print('=' * 60)

    # Step 1
    df, X_scaled, scaler, feature_cols = step1_prepare_data()

    # Step 2
    optimal_k = step2_find_optimal_k(X_scaled)

    # Step 3
    gmm, labels, probabilities = step3_train_gmm(X_scaled, optimal_k)

    # Step 4
    profile = step4_profile_clusters(df, labels, feature_cols)

    # Step 5
    step5_visualize(df, X_scaled, labels, probabilities, profile, feature_cols)

    # Step 6
    step6_compare_rfm(df, labels, profile)

    # Step 7
    step7_recommendations(profile)

    # Summary
    print('\n' + '='*60)
    print('PIPELINE COMPLETE')
    print('='*60)
    print(f'  Figures saved to: {FIG_DIR}')
    print(f'  Data saved to:    {DATA_DIR}')
    print(f'  Optimal clusters: {optimal_k}')
    print(f'  Total customers:  {len(df)}')


main()